# Compare DINO+SAM2 vs SAM3 on 20 Random Frames

Minimal experiment notebook for comparing the current auto-label proposal pipeline against a SAM3 pipeline on the same sampled frames and text prompts.

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import importlib.util
import random
import sys
import traceback
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    torch = None
    DEVICE = "cpu"

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "scripts" / "auto_label" / "auto_label_gui.py").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError(
        "Could not find repo root. Start Jupyter from prompt_video_segmenter or set REPO_ROOT manually."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FRAME_DIR = REPO_ROOT / "data" / "auto_label_demo" / "frames"
OUTPUT_DIR = REPO_ROOT / "outputs" / "compare_dino_sam2_vs_sam3_20frames"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_SAMPLES = 20
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

DINO_BOX_THRESHOLD = 0.25
DINO_TEXT_THRESHOLD = 0.25

# Optional: point this to a local Python file that defines:
#   load_sam3_model(device: str) -> Any
#   predict_sam3(model: Any, image_path: Path, prompts: list[str]) -> list[dict]
SAM3_ADAPTER_PY = str(REPO_ROOT / "notebooks" / "owlv2_sam2_adapter_for_compare.py")

print(f"Repo root : {REPO_ROOT}")
print(f"Frame dir : {FRAME_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Device    : {DEVICE}")

## 2. Locate frames and sample 20 images

In [ ]:
def list_images(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )


def find_frame_dir() -> Path:
    candidates = [
        FRAME_DIR,
        REPO_ROOT / "data" / "auto_label_demo" / "frames",
        REPO_ROOT / "data" / "frames",
        REPO_ROOT / "outputs" / "frames",
    ]
    scored = [(len(list_images(path)), path) for path in candidates]
    scored = [(count, path) for count, path in scored if count > 0]
    if scored:
        return max(scored, key=lambda item: item[0])[1]

    search_roots = [REPO_ROOT / "data", REPO_ROOT / "outputs"]
    discovered: list[tuple[int, Path]] = []
    for base in search_roots:
        if not base.exists():
            continue
        for directory in [base, *[p for p in base.rglob("*") if p.is_dir()]]:
            count = len(list_images(directory))
            if count > 0:
                discovered.append((count, directory))
    if not discovered:
        raise FileNotFoundError("No extracted frame directory found. Set FRAME_DIR manually in section 1.")
    return max(discovered, key=lambda item: item[0])[1]


FRAME_DIR = find_frame_dir()
all_frames = list_images(FRAME_DIR)
if len(all_frames) < N_SAMPLES:
    raise ValueError(f"Only found {len(all_frames)} images in {FRAME_DIR}; need {N_SAMPLES}.")

rng = random.Random(RANDOM_SEED)
sampled_frames = rng.sample(all_frames, N_SAMPLES)

print(f"Using FRAME_DIR: {FRAME_DIR}")
print(f"Found {len(all_frames)} frames; sampled {len(sampled_frames)}.")
for i, path in enumerate(sampled_frames[:5]):
    print(f"{i:02d}: {path}")

## 3. Load text prompts from existing autolabel GUI

The list below is copied from the current `scripts/auto_label/auto_label_gui.py` prompt text box default.

In [ ]:
# Copied from AutoLabelWindow._build_config_panel() in scripts/auto_label/auto_label_gui.py.
TEXT_PROMPTS = [
    "hand", "glove",
    "pot", "pan", "lid", "cookware", "tray", "kettle",
    "bowl", "plate", "cup", "glass", "bottle", "jar",
    "container", "box", "package", "bag", "carton", "can",
    "knife", "fork", "spoon", "spatula", "tongs", "ladle", "whisk", "peeler", "scissors", "cutting board",
    "pasta", "noodles", "rice", "bread", "vegetable", "fruit", "meat", "fish", "egg", "cheese",
    "ingredient", "food", "dry food", "liquid", "water", "milk", "sauce", "oil", "powder", "sugar", "salt",
    "sink", "faucet", "stove", "cooktop", "oven", "microwave", "fridge",
    "drawer", "cabinet", "countertop", "table", "rack", "sponge", "towel",
]

print(f"Loaded {len(TEXT_PROMPTS)} prompts:")
print(", ".join(TEXT_PROMPTS))

## 4. Load DINO + SAM2

In [ ]:
dino_sam2_backend = None
dino_sam2_load_error = ""

try:
    from scripts.auto_label.generate_mask_proposals import GroundingDINOSAM2Backend

    dino_sam2_backend = GroundingDINOSAM2Backend(
        confidence_threshold=DINO_BOX_THRESHOLD,
        box_threshold=DINO_BOX_THRESHOLD,
        text_threshold=DINO_TEXT_THRESHOLD,
        device=DEVICE,
    )
    print(f"DINO+SAM2 loaded: {dino_sam2_backend.name}")
except Exception as exc:
    dino_sam2_load_error = f"{type(exc).__name__}: {exc}"
    print("DINO+SAM2 failed to load:")
    print(dino_sam2_load_error)
    traceback.print_exc(limit=2)


def run_dino_sam2_on_image(image_path: Path, prompts: list[str]) -> dict:
    if dino_sam2_backend is None:
        raise RuntimeError(dino_sam2_load_error or "DINO+SAM2 backend was not loaded.")
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")
    instances = dino_sam2_backend.generate(image_bgr, prompts, image_path.stem)
    return {"instances": instances, "error": ""}

## 5. Load comparison adapter

By default this notebook uses `notebooks/owlv2_sam2_adapter_for_compare.py`, which runs OWLv2 text-conditioned box detection and then feeds those boxes into the project SAM2 segmenter. The adapter keeps the old SAM3 function names so the notebook structure stays minimal.

In [ ]:
sam3_model = None
sam3_adapter = None
sam3_load_error = ""


def load_sam3_from_adapter(adapter_path: Path):
    spec = importlib.util.spec_from_file_location("local_sam3_adapter", adapter_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not import SAM3 adapter: {adapter_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    if not hasattr(module, "load_sam3_model") or not hasattr(module, "predict_sam3"):
        raise AttributeError("Adapter must define load_sam3_model(device) and predict_sam3(model, image_path, prompts).")
    return module, module.load_sam3_model(DEVICE)


try:
    if SAM3_ADAPTER_PY:
        sam3_adapter, sam3_model = load_sam3_from_adapter(Path(SAM3_ADAPTER_PY))
        print(f"SAM3 loaded via adapter: {SAM3_ADAPTER_PY}")
    else:
        raise RuntimeError(
            "SAM3 adapter not configured. Set SAM3_ADAPTER_PY to a local adapter file once SAM3 is installed. "
            "The rest of the notebook will still run and mark SAM3 results as missing."
        )
except Exception as exc:
    sam3_load_error = f"{type(exc).__name__}: {exc}"
    print("SAM3 not available:")
    print(sam3_load_error)


def run_sam3_on_image(image_path: Path, prompts: list[str]) -> dict:
    if sam3_adapter is None or sam3_model is None:
        raise RuntimeError(sam3_load_error or "SAM3 was not loaded.")
    instances = sam3_adapter.predict_sam3(sam3_model, image_path, prompts)
    return {"instances": instances, "error": ""}

## 6. Run inference on sampled frames

In [ ]:
def safe_run(method_name: str, fn, image_path: Path, prompts: list[str]) -> dict:
    try:
        result = fn(image_path, prompts)
        result.setdefault("instances", [])
        result.setdefault("error", "")
        return result
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        print(f"[{method_name}] failed on {image_path.name}: {error}")
        return {"instances": [], "error": error}


results = []
for idx, image_path in enumerate(sampled_frames):
    print(f"[{idx + 1:02d}/{len(sampled_frames)}] {image_path}")
    dino_result = safe_run("DINO_SAM2", run_dino_sam2_on_image, image_path, TEXT_PROMPTS)
    sam3_result = safe_run("SAM3", run_sam3_on_image, image_path, TEXT_PROMPTS)
    results.append({
        "frame_path": image_path,
        "DINO_SAM2": dino_result,
        "SAM3": sam3_result,
    })

print("Done.")

## 7. Visualize side-by-side comparisons

In [ ]:
def load_rgb(image_path: Path) -> np.ndarray:
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def color_for_index(index: int) -> tuple[float, float, float]:
    cmap = plt.get_cmap("tab20")
    return tuple(cmap(index % 20)[:3])


def normalize_instances(instances: list[dict]) -> list[dict]:
    rows = []
    for inst in instances or []:
        rows.append({
            "label": inst.get("label", inst.get("class", inst.get("name", "object"))),
            "score": inst.get("confidence", inst.get("score", None)),
            "bbox": inst.get("bbox_xyxy", inst.get("bbox", None)),
            "mask": inst.get("mask", None),
        })
    return rows


def draw_result(ax, image_rgb: np.ndarray, instances: list[dict], title: str, error: str = "") -> None:
    ax.imshow(image_rgb)
    ax.set_title(title, fontsize=11)
    ax.axis("off")
    if error:
        ax.text(8, 22, error[:120], color="white", fontsize=8, bbox={"facecolor": "red", "alpha": 0.7})
        return

    for i, inst in enumerate(normalize_instances(instances)):
        color = color_for_index(i)
        mask = inst.get("mask")
        if mask is not None:
            mask_arr = np.asarray(mask).astype(bool)
            if mask_arr.shape[:2] == image_rgb.shape[:2]:
                overlay = np.zeros((*mask_arr.shape, 4), dtype=float)
                overlay[mask_arr, :3] = color
                overlay[mask_arr, 3] = 0.35
                ax.imshow(overlay)

        bbox = inst.get("bbox")
        if bbox is not None and len(bbox) >= 4:
            x1, y1, x2, y2 = [float(v) for v in bbox[:4]]
            rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=1.5, edgecolor=color)
            ax.add_patch(rect)
            label = str(inst.get("label", "object"))
            score = inst.get("score")
            if score is not None:
                label = f"{label} {float(score):.2f}"
            ax.text(x1, max(0, y1 - 3), label, color="white", fontsize=7, bbox={"facecolor": color, "alpha": 0.8, "pad": 1})


comparison_paths = []
for idx, row in enumerate(results):
    image_rgb = load_rgb(row["frame_path"])
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
    draw_result(axes[0], image_rgb, [], "Original")
    draw_result(
        axes[1], image_rgb,
        row["DINO_SAM2"]["instances"],
        f"DINO + SAM2 ({len(row['DINO_SAM2']['instances'])})",
        row["DINO_SAM2"].get("error", ""),
    )
    draw_result(
        axes[2], image_rgb,
        row["SAM3"]["instances"],
        f"SAM3 ({len(row['SAM3']['instances'])})",
        row["SAM3"].get("error", ""),
    )
    out_path = OUTPUT_DIR / f"comparison_{idx:03d}.png"
    fig.savefig(out_path, dpi=160)
    comparison_paths.append(out_path)
    plt.show()

print(f"Saved {len(comparison_paths)} comparison images to {OUTPUT_DIR}")

## 8. Save summary CSV

In [ ]:
summary_rows = []
for row, comparison_path in zip(results, comparison_paths):
    summary_rows.append({
        "frame_path": str(row["frame_path"]),
        "dino_sam2_num_instances": len(row["DINO_SAM2"].get("instances", [])),
        "sam3_num_instances": len(row["SAM3"].get("instances", [])),
        "dino_sam2_error": row["DINO_SAM2"].get("error", ""),
        "sam3_error": row["SAM3"].get("error", ""),
        "comparison_output_path": str(comparison_path),
    })

summary_df = pd.DataFrame(summary_rows)
summary_csv = OUTPUT_DIR / "summary.csv"
summary_df.to_csv(summary_csv, index=False)
display(summary_df)
print(f"Saved: {summary_csv}")

## 9. Create manual review template

In [ ]:
review_rows = []
for row in results:
    for method_key, method_name in [("DINO_SAM2", "DINO_SAM2"), ("SAM3", "SAM3")]:
        review_rows.append({
            "frame_path": str(row["frame_path"]),
            "method": method_name,
            "num_instances": len(row[method_key].get("instances", [])),
            "mask_boundary_quality": "",
            "missed_objects": "",
            "wrong_objects": "",
            "hand_object_separation": "",
            "overall_score": "",
            "notes": "",
        })

manual_review_df = pd.DataFrame(review_rows)
manual_review_csv = OUTPUT_DIR / "manual_review_template.csv"
manual_review_df.to_csv(manual_review_csv, index=False)
display(manual_review_df.head())
print(f"Saved: {manual_review_csv}")

## 10. Notes and next steps

- DINO+SAM2 reuses the existing `GroundingDINOSAM2Backend` from `scripts/auto_label/generate_mask_proposals.py`.
- The prompt list is copied from the current AutoLabel GUI default prompt text.
- The third column is adapter-based. By default it runs OWLv2 boxes plus SAM2 masks through `notebooks/owlv2_sam2_adapter_for_compare.py`.
- Use `manual_review_template.csv` to score boundary quality, missed objects, wrong objects, hand/object separation, and overall visual quality.